# Bài 8: Phân cụm K-Means – Thực hành trên Google Colab


In [ ]:
import os, shutil, zipfile
if not os.path.isdir("data"):
    try:
        from google.colab import files
        print("Hay chon tep Lab08_data.zip (hoac cac tep .csv cua bai)")
        up = files.upload()
    except ImportError:
        raise FileNotFoundError("Khong thay thu muc data/, hay dat no canh notebook")
    os.makedirs("data", exist_ok=True)
    for ten in up:
        if ten.endswith(".zip"):
            zipfile.ZipFile(ten).extractall(".")
        elif ten.endswith(".csv"):
            shutil.move(ten, os.path.join("data", ten))
print(sorted(os.listdir("data")))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/khach_hang.csv")
print(df.shape)
df.head()

In [ ]:
P = np.array([[1.0, 1.0], [1.5, 2.0], [3.0, 4.0], [5.0, 7.0],
              [3.5, 5.0], [4.5, 5.0], [3.5, 4.5]])
TEN = ["A", "B", "C", "D", "E", "F", "G"]

def k_means(X, tam, so_vong_toi_da=10):
    """K-Means don gian: gan diem vao tam gan nhat roi doi tam."""
    for vong in range(1, so_vong_toi_da + 1):
        # Buoc gan: khoang cach tu moi diem toi moi tam
        kc = np.sqrt(((X[:, None, :] - tam[None, :, :]) ** 2).sum(axis=2))
        nhan = kc.argmin(axis=1)
        print(f"Vong {vong}: tam = {np.round(tam, 2).tolist()}")
        for k in range(len(tam)):
            ds = [TEN[i] for i in range(len(X)) if nhan[i] == k]
            print(f"  cum {k + 1}: {', '.join(ds)}")
        # Buoc doi tam: trung binh cac diem trong moi cum
        tam_moi = np.array([X[nhan == k].mean(axis=0) for k in range(len(tam))])
        if np.allclose(tam_moi, tam):
            print("  Tam khong doi nua, dung.")
            return tam, nhan
        tam = tam_moi
    return tam, nhan

tam, nhan = k_means(P, P[[0, 1]].copy())
print()
inertia = ((P - tam[nhan]) ** 2).sum()
print(f"Inertia tu tinh: {inertia:.4f}")

km = KMeans(n_clusters=2, init=P[[0, 1]], n_init=1).fit(P)
print(f"Inertia cua scikit-learn: {km.inertia_:.4f}")
print("Tam cua scikit-learn:", np.round(km.cluster_centers_, 2).tolist())

In [ ]:
print(df.describe().round(1).to_string())
plt.figure(figsize=(5, 3.5))
plt.scatter(df["thu_nhap"], df["diem_chi_tieu"], s=14, c="gray")
plt.xlabel("Thu nhap (trieu dong/thang)"); plt.ylabel("Diem chi tieu"); plt.title("Khong co nhan"); plt.show()

In [ ]:
X = df[["thu_nhap", "diem_chi_tieu"]]
sc = StandardScaler().fit(X)
Xs = sc.transform(X)

km = KMeans(n_clusters=5, n_init=10, random_state=0).fit(Xs)
df["cum"] = km.labels_

# Tam cum dang o thang da chuan hoa -> doi nguoc ve don vi goc de doc
tam = pd.DataFrame(sc.inverse_transform(km.cluster_centers_), columns=["thu_nhap", "diem_chi_tieu"])
tam["so_khach"] = np.bincount(km.labels_)
tam["tuoi_tb"] = df.groupby("cum")["tuoi"].mean()
print("Tam cum, don vi goc:")
print(tam.round(1).to_string())
print(f"\nInertia: {km.inertia_:.2f}\n")

# Khach moi: chuan hoa bang DUNG bo chuan hoa cu, roi tim tam gan nhat
moi = pd.DataFrame({"thu_nhap": [29.0, 9.0], "diem_chi_tieu": [85, 75]})
cum_moi = km.predict(sc.transform(moi))
for i in range(len(moi)):
    print(f"Khach moi {i + 1}: thu nhap {moi.iloc[i, 0]}, chi tieu {moi.iloc[i, 1]} -> cum {cum_moi[i]}")

In [ ]:
plt.figure(figsize=(5.5, 4))
plt.scatter(df["thu_nhap"], df["diem_chi_tieu"], c=df["cum"], cmap="tab10", s=16)
plt.scatter(tam["thu_nhap"], tam["diem_chi_tieu"], c="k", marker="X", s=150)
plt.xlabel("Thu nhap (trieu dong/thang)"); plt.ylabel("Diem chi tieu"); plt.title("K-Means voi K = 5"); plt.show()

In [ ]:
print(" K  inertia  giam so voi K-1  silhouette")
truoc = None
ks, ines, sils = [], [], []
for k in range(1, 11):
    km_k = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Xs)
    giam = "" if truoc is None else f"{truoc - km_k.inertia_:8.1f}"
    sil = "" if k == 1 else f"{silhouette_score(Xs, km_k.labels_):.3f}"
    print(f"{k:2d} {km_k.inertia_:8.1f} {giam:>15s} {sil:>11s}")
    truoc = km_k.inertia_
    ks.append(k); ines.append(km_k.inertia_); sils.append(np.nan if k == 1 else silhouette_score(Xs, km_k.labels_))

fig, axs = plt.subplots(1, 2, figsize=(9, 3.2))
axs[0].plot(ks, ines, "o-"); axs[0].set_title("Phuong phap khuyu tay"); axs[0].set_xlabel("So cum K"); axs[0].set_ylabel("Inertia")
axs[1].plot(ks, sils, "o-"); axs[1].set_title("Chi so silhouette"); axs[1].set_xlabel("So cum K")
plt.tight_layout(); plt.show()

In [ ]:
# Bay 1: thu nhap ghi bang nghin dong, khong chuan hoa
X_nghin = X.copy()
X_nghin["thu_nhap"] = X_nghin["thu_nhap"] * 1000
km_tho = KMeans(n_clusters=5, n_init=10, random_state=0).fit(X_nghin)
tam_tho = km_tho.cluster_centers_.copy()
tam_tho[:, 0] = tam_tho[:, 0] / 1000
print("Bay 1: khong chuan hoa, tam cum (thu nhap trieu, chi tieu):")
for t in tam_tho[np.argsort(tam_tho[:, 0])]:
    print(f"  ({t[0]:5.1f}, {t[1]:5.1f})")
print()

# Bay 2: moi lan chi khoi tao ngau nhien mot lan
ket = sorted(round(KMeans(n_clusters=5, init="random", n_init=1, random_state=rs).fit(Xs).inertia_, 1)
             for rs in range(20))
print("Bay 2: inertia cua 20 lan khoi tao ngau nhien:")
print("  ", ket[:10]); print("  ", ket[10:])
tot = KMeans(n_clusters=5, n_init=10, random_state=0).fit(Xs)
print(f"  Voi n_init=10: inertia {tot.inertia_:.1f}")
print()

# Bay 3: cum khong tron, du lieu hai vang trang
tr = pd.read_csv("data/hai_vang_trang.csv")
nhan_tr = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(tr[["x1", "x2"]])
khop = (nhan_tr == tr["nhan"]).mean()
print(f"Bay 3: hai vang trang, ty le khop nhan that {max(khop, 1 - khop):.4f}")

In [ ]:
from sklearn.datasets import load_sample_image
anh = load_sample_image("china.jpg")
cao, rong, _ = anh.shape
diem = anh.reshape(-1, 3) / 255.0
so_mau = len(np.unique(anh.reshape(-1, 3), axis=0))
print(f"Anh {cao} x {rong}, co {cao * rong} diem anh, {so_mau} mau")
print()

rng = np.random.default_rng(0)
mau = diem[rng.choice(len(diem), 5000, replace=False)]   # hoc tren 5000 diem cho nhanh

print("  K  so bit/diem  nho hon  sai lech trung binh")
anh_nen = {}
for k in [4, 16, 64]:
    km_a = KMeans(n_clusters=k, n_init=4, random_state=0).fit(mau)
    nhan_a = km_a.predict(diem)
    moi_a = km_a.cluster_centers_[nhan_a]
    bit = int(np.ceil(np.log2(k)))
    sai = np.abs(moi_a - diem).mean() * 255
    anh_nen[k] = moi_a.reshape(cao, rong, 3)
    print(f" {k:2d}  {bit:6d}  {24 / bit:6.1f} lan  {sai:6.2f} muc sang")

fig, axs = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, (t, a) in zip(axs, [("Anh goc", anh / 255.0), ("Chi con 16 mau", anh_nen[16]), ("Chi con 4 mau", anh_nen[4])]):
    ax.imshow(a); ax.set_title(t); ax.axis("off")
plt.show()

---
# Bài tập


## Bài tập 1: Tự tính silhouette


In [ ]:
def silhouette_mot_diem(X, nhan, i):
    kc = np.sqrt(((X - X[i]) ** 2).sum(axis=1))        # khoang cach tu i toi moi diem
    cung = (nhan == nhan[i]); cung[i] = False           # cung cum, tru chinh no
    if cung.sum() == 0:
        return 0.0, 0.0, 0.0
    a = kc[cung].mean()
    b = min(kc[nhan == k].mean() for k in np.unique(nhan) if k != nhan[i])
    return a, b, (b - a) / max(a, b)

nhan5 = KMeans(n_clusters=5, n_init=10, random_state=0).fit_predict(Xs)
s_tat_ca = np.array([silhouette_mot_diem(Xs, nhan5, i)[2] for i in range(len(Xs))])
print(f"Trung binh tu tinh : {s_tat_ca.mean():.4f}")
print(f"silhouette_score   : {silhouette_score(Xs, nhan5):.4f}")
i_min = int(s_tat_ca.argmin())
print(f"Khach co s nho nhat: so {i_min}, s = {s_tat_ca[i_min]:.3f},",
      df.loc[i_min, ["tuoi", "thu_nhap", "diem_chi_tieu"]].to_dict())

plt.figure(figsize=(5.5, 4))
plt.scatter(X["thu_nhap"], X["diem_chi_tieu"], c=nhan5, cmap="tab10", s=14)
plt.scatter(X.loc[i_min, "thu_nhap"], X.loc[i_min, "diem_chi_tieu"], s=200, facecolors="none", edgecolors="k", linewidths=2)
plt.title("Khach co s nho nhat (khoanh den)"); plt.show()

## Bài tập 2: Thêm cột tuổi


In [ ]:
cot3 = ["tuoi", "thu_nhap", "diem_chi_tieu"]
sc3 = StandardScaler().fit(df[cot3])
X3 = sc3.transform(df[cot3])
for k in range(2, 11):
    s = silhouette_score(X3, KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X3))
    print(f"K = {k:2d}  silhouette {s:.3f}")

k_tot = max(range(2, 11), key=lambda k: silhouette_score(X3, KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X3)))
km3 = KMeans(n_clusters=k_tot, n_init=10, random_state=0).fit(X3)
tam3 = pd.DataFrame(sc3.inverse_transform(km3.cluster_centers_), columns=cot3)
tam3["so_khach"] = np.bincount(km3.labels_)
print(f"\nK tot nhat = {k_tot}. Tam cum (don vi goc):")
print(tam3.round(1).to_string())

## Bài tập 3: K-Means++ và khởi tạo ngẫu nhiên


In [ ]:
for cach in ["random", "k-means++"]:
    ine = [KMeans(n_clusters=5, init=cach, n_init=1, random_state=rs).fit(Xs).inertia_ for rs in range(50)]
    tot_ = sum(round(v, 1) == 23.0 for v in ine)
    print(f"{cach:10s}: {tot_}/50 lan dat inertia 23.0, te nhat {max(ine):.1f}")

## Bài tập 4: Lưu ảnh đã nén


In [ ]:
km16 = KMeans(n_clusters=16, n_init=4, random_state=0).fit(mau)
anh_moi = km16.cluster_centers_[km16.predict(diem)].reshape(cao, rong, 3)
plt.imsave("anh_goc.png", anh)
plt.imsave("anh_16_mau.png", anh_moi)
print("Anh goc  :", os.path.getsize("anh_goc.png") // 1024, "KB")
print("Anh 16 mau:", os.path.getsize("anh_16_mau.png") // 1024, "KB")

## Bài tập 5: DBSCAN trên hai vầng trăng


In [ ]:
from sklearn.cluster import DBSCAN
tr = pd.read_csv("data/hai_vang_trang.csv")
Xtr_ = tr[["x1", "x2"]]
for e in [0.2, 0.3, 0.4]:
    nh = DBSCAN(eps=e, min_samples=5).fit_predict(Xtr_)
    so_cum = len(set(nh)) - (1 if -1 in nh else 0)
    nhieu = int((nh == -1).sum())
    mask = nh != -1
    khop = (nh[mask] == tr["nhan"][mask]).mean() if mask.any() else float("nan")
    print(f"eps = {e}: {so_cum} cum, {nhieu} diem nhieu, khop nhan that tren diem khong nhieu {max(khop, 1 - khop):.4f}")

## Bài tập 6: Phân cụm chữ số viết tay


In [ ]:
from sklearn.datasets import load_digits
Xd, yd = load_digits(return_X_y=True)
cum_d = KMeans(n_clusters=10, n_init=10, random_state=0).fit_predict(Xd)
pd.crosstab(pd.Series(yd, name="chu so that"), pd.Series(cum_d, name="cum"))